In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [2]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid

In [3]:
warnings.filterwarnings("ignore")
np.random.seed(42)

In [4]:
DATA_DIR = "."
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"  Found {len(csv_files)} CSV files")
 

  Found 17 CSV files


In [6]:
def load_ts_file(path):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["source"] = os.path.basename(path).replace(".csv", "")
    name = os.path.basename(path)
    if   "cpu"     in name: df["metric"] = "cpu"
    elif "disk"    in name: df["metric"] = "disk"
    elif "network" in name: df["metric"] = "network"
    elif "elb"     in name: df["metric"] = "elb"
    elif "rds"     in name: df["metric"] = "rds"
    else:                   df["metric"] = "other"
    return df
 

In [7]:
frames = [load_ts_file(f) for f in csv_files]
raw    = pd.concat(frames, ignore_index=True)

In [8]:
print(f"  Total rows       : {len(raw):,}")
print(f"  Date range       : {raw['timestamp'].min()} to {raw['timestamp'].max()}")
print(f"  Sources          : {raw['source'].nunique()}")
print(f"  Metric categories: {raw['metric'].unique().tolist()}")
print(f"  Missing values   : {raw['value'].isna().sum()}")
print(f"\n  Per-metric summary:")
print(raw.groupby("metric")["value"].describe().round(3).to_string())

  Total rows       : 67,740
  Date range       : 2013-10-09 16:25:00 to 2014-04-24 00:39:00
  Sources          : 17
  Metric categories: ['cpu', 'disk', 'network', 'elb', 'other']
  Missing values   : 0

  Per-metric summary:
           count          mean           std     min     25%        50%         75%           max
metric                                                                                           
cpu      40320.0  2.192700e+01  2.986400e+01   0.062   0.134      5.860      35.798  9.989800e+01
disk      8762.0  1.152824e+07  6.189900e+07   0.000   0.000      0.000       0.000  8.639640e+08
elb       4032.0  6.183700e+01  5.666500e+01   1.000  15.000     48.000      89.000  6.560000e+02
network   8762.0  3.267548e+05  3.185039e+06  42.000  68.400  16794.800  234791.750  2.451260e+08
other     5864.0  9.783166e+05  2.813144e+06   0.000  33.333     33.446      35.953  6.151940e+07


STEP 2 : FEATURE ENGINEERING

In [9]:
def engineer_features(df, window=12):
    g = df.copy().sort_values("timestamp").reset_index(drop=True)
    v = g["value"]
 
    roll               = v.rolling(window, min_periods=1)
    g["rolling_mean"]  = roll.mean()
    g["rolling_std"]   = roll.std().fillna(0)
    g["rolling_min"]   = roll.min()
    g["rolling_max"]   = roll.max()
    g["rolling_range"] = g["rolling_max"] - g["rolling_min"]
    g["z_score"]       = (v - g["rolling_mean"]) / (g["rolling_std"] + 1e-9)
 
    for lag in [1, 2, 3]:
        g[f"lag_{lag}"] = v.shift(lag).bfill()
    g["diff_1"] = v.diff(1).fillna(0)
    g["diff_2"] = v.diff(2).fillna(0)
 
    ts = g["timestamp"]
    g["hour_sin"]  = np.sin(2 * np.pi * ts.dt.hour / 24)
    g["hour_cos"]  = np.cos(2 * np.pi * ts.dt.hour / 24)
    g["dow_sin"]   = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    g["dow_cos"]   = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    return g

In [10]:
feature_frames = [engineer_features(grp) for _, grp in raw.groupby("source")]
data = pd.concat(feature_frames, ignore_index=True)

In [11]:
FEATURE_COLS = [
    "value", "rolling_mean", "rolling_std", "rolling_min",
    "rolling_max", "rolling_range", "z_score",
    "lag_1", "lag_2", "lag_3", "diff_1", "diff_2",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]

In [12]:
X = data[FEATURE_COLS].copy()
print(f"  Feature matrix shape : {X.shape}")
print(f"  Features used        : {FEATURE_COLS}")
print(f"  NaNs after engineering: {X.isna().sum().sum()}")

  Feature matrix shape : (67740, 16)
  Features used        : ['value', 'rolling_mean', 'rolling_std', 'rolling_min', 'rolling_max', 'rolling_range', 'z_score', 'lag_1', 'lag_2', 'lag_3', 'diff_1', 'diff_2', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
  NaNs after engineering: 0


STEP 3 : PREPROCESSING & SCALING

In [13]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"  StandardScaler applied -> shape {X_scaled.shape}")
print(f"  Feature means (first 4): {X_scaled.mean(axis=0).round(3)[:4]}")
print(f"  Feature stds  (first 4): {X_scaled.std(axis=0).round(3)[:4]}")

  StandardScaler applied -> shape (67740, 16)
  Feature means (first 4): [-0. -0.  0.  0.]
  Feature stds  (first 4): [1. 1. 1. 1.]


STEP 4 : HYPERPARAMETER TUNING + CROSS-VALIDATION

In [29]:
PARAM_GRID = {
    "n_estimators": [50, 100],
    "max_samples": [256, 512],
    "contamination": [0.01, 0.03, 0.05],
    "max_features": [0.5, 0.8],
    "bootstrap": [False]
}

In [32]:
from sklearn.model_selection import ParameterSampler
N_FOLDS = 3
N_ITER = 25              
SUBSAMPLE_N = 4000      
MAX_TIME_PER_CONFIG = 6
param_list = list(ParameterSampler(PARAM_GRID, n_iter=N_ITER, random_state=42))

In [33]:
X_sub = X_scaled[:SUBSAMPLE_N].astype(np.float32)

In [24]:
rng     = np.random.default_rng(42)
sub_idx = rng.choice(len(X_scaled), size=SUBSAMPLE_N, replace=False)
sub_idx.sort()                          # ← keeps time order intact
X_sub   = X_scaled[sub_idx]

In [34]:
def time_folds(n, n_folds=3):
    fold = n // n_folds
    for k in range(n_folds):
        val_start = k * fold
        val_end = val_start + fold if k < n_folds - 1 else n

        if val_start == 0:
            continue

        train_idx = list(range(0, val_start))
        val_idx = list(range(val_start, val_end))
        yield train_idx, val_idx


In [35]:
def evaluate_params_fast(params, X, n_folds=3):
    ratios, scores_all = [], []

    for tr, val in time_folds(len(X), n_folds):
        model = IsolationForest(
            n_jobs=1,            # ⚠️ CPU SAFE
            random_state=42,
            **params
        )

        model.fit(X[tr])

        preds = model.predict(X[val])
        scores = model.score_samples(X[val])

        ratios.append((preds == -1).mean())
        scores_all.append(np.abs(scores).mean())

    return np.mean(ratios), np.std(ratios), np.mean(scores_all)


In [37]:
import time
results = []
best_composite = -np.inf
best_params = None

print("\n🚀 Running optimized tuning...\n")

for i, params in enumerate(param_list):
    start_time = time.time()

    try:
        mean_r, std_r, mean_s = evaluate_params_fast(params, X_sub, N_FOLDS)

        # 🚫 Skip bad configs early
        if mean_r > 0.15:
            continue

        composite = mean_s / (1.0 + std_r)

        results.append({
            **params,
            "mean_ratio": round(mean_r, 4),
            "std_ratio": round(std_r, 4),
            "mean_score": round(mean_s, 4),
            "composite": round(composite, 4)
        })

        # Track best
        if composite > best_composite:
            best_composite = composite
            best_params = params.copy()

    except Exception as e:
        print(f"Skipping error config: {params}")
        continue

    # ⏱️ Stop slow configs
    if time.time() - start_time > MAX_TIME_PER_CONFIG:
        print(f"⏱️ Slow config skipped: {params}")

    # 📊 Progress
    print(f"[{i+1:>2}/{len(param_list)}] best={best_composite:.4f}")



🚀 Running optimized tuning...

[ 1/24] best=0.5324
[ 2/24] best=0.5332
[ 3/24] best=0.5332
[ 4/24] best=0.5332
[ 5/24] best=0.5380
[ 6/24] best=0.5380
[ 7/24] best=0.5380
[ 8/24] best=0.5380
[ 9/24] best=0.5380
[10/24] best=0.5380
[11/24] best=0.5380
[12/24] best=0.5380
[13/24] best=0.5380
[14/24] best=0.5380
[15/24] best=0.5380
[16/24] best=0.5380
[17/24] best=0.5380
[18/24] best=0.5380
[19/24] best=0.5380
[20/24] best=0.5380
[21/24] best=0.5380
[22/24] best=0.5380
[23/24] best=0.5380
[24/24] best=0.5380


In [38]:
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    results_df = results_df.sort_values("composite", ascending=False)

    print("\nTop 5 configs:\n")
    print(results_df.head(5).to_string(index=False))

    print(f"\nBest params: {best_params}")
    print(f"Best composite: {best_composite:.4f}")

else:
    print(" No valid results found")


Top 5 configs:

 n_estimators  max_samples  max_features  contamination  bootstrap  mean_ratio  std_ratio  mean_score  composite
           50          256           0.8           0.01      False      0.0262     0.0060      0.5413     0.5380
          100          256           0.8           0.01      False      0.0304     0.0079      0.5402     0.5360
           50          256           0.8           0.03      False      0.0637     0.0143      0.5413     0.5337
          100          256           0.5           0.01      False      0.0315     0.0053      0.5360     0.5332
          100          512           0.8           0.01      False      0.0195     0.0007      0.5330     0.5326

Best params: {'n_estimators': 50, 'max_samples': 256, 'max_features': 0.8, 'contamination': 0.01, 'bootstrap': False}
Best composite: 0.5380


In [42]:
results = []
 
print(f"\n STEP 4: HYPERPARAMETER TUNING + CROSS-VALIDATION")
print(f"  Grid size      : {total} combos")
print(f"  CV rows        : {SUBSAMPLE_N:,} of {len(X_scaled):,} "
      f"({100*SUBSAMPLE_N/len(X_scaled):.1f}%)")
print(f"  Folds          : {N_FOLDS} (time-aware, chronological)")
print(f"  max_samples    : integer → each tree sees at most 512 rows")
print(f"  dtype          : float32 → ~50% less RAM than float64")
print(f"  Running ...")
 
for i, params in enumerate(param_list):
 
    mean_r, std_r, mean_s = evaluate_params_fast(params, X_sub, N_FOLDS)
 
    # Composite score:
    #   Numerator  → higher mean |score| = model is more confident
    #   Denominator → penalise high std (unstable detection across folds)
    composite = mean_s / (1.0 + std_r)
 
    results.append({
        **params,
        "mean_ratio" : round(mean_r,    4),
        "std_ratio"  : round(std_r,     4),
        "mean_score" : round(mean_s,    4),
        "composite"  : round(composite, 4),
    })
 
    # Track best without touching the results list
    if composite > best_composite:
        best_composite = composite
        best_params    = params.copy()
 
    # Progress — print every 5 combos
    if (i + 1) % 5 == 0 or (i + 1) == total:
        print(f"  [{i+1:>3}/{total}]  best composite={best_composite:.4f}  "
              f"→ {best_params}")
 
# ── RESULTS TABLE ────────────────────────────────────────────────────
results_df = (
    pd.DataFrame(results)
    .sort_values("composite", ascending=False)
    .reset_index(drop=True)
)
 
print("\n  Top 5 parameter sets:")
print(
    results_df[[
        "n_estimators", "max_samples", "contamination",
        "max_features", "composite", "mean_ratio", "std_ratio"
    ]].head(5).to_string(index=False)
)
 
# Strip numpy scalar wrappers so sklearn accepts the dict cleanly
best_params = {k: (int(v) if isinstance(v, (np.integer,)) else
                   float(v) if isinstance(v, (np.floating,)) else v)
               for k, v in best_params.items()}
 
print(f"\n  Best params : {best_params}")
print(f"  Best composite score : {best_composite:.4f}")
 


 STEP 4: HYPERPARAMETER TUNING + CROSS-VALIDATION
  Grid size      : 24 combos
  CV rows        : 4,000 of 67,740 (5.9%)
  Folds          : 3 (time-aware, chronological)
  max_samples    : integer → each tree sees at most 512 rows
  dtype          : float32 → ~50% less RAM than float64
  Running ...
  [  5/24]  best composite=0.5380  → {'n_estimators': 50, 'max_samples': 256, 'max_features': 0.8, 'contamination': 0.01, 'bootstrap': False}
  [ 10/24]  best composite=0.5380  → {'n_estimators': 50, 'max_samples': 256, 'max_features': 0.8, 'contamination': 0.01, 'bootstrap': False}
  [ 15/24]  best composite=0.5380  → {'n_estimators': 50, 'max_samples': 256, 'max_features': 0.8, 'contamination': 0.01, 'bootstrap': False}
  [ 20/24]  best composite=0.5380  → {'n_estimators': 50, 'max_samples': 256, 'max_features': 0.8, 'contamination': 0.01, 'bootstrap': False}
  [ 24/24]  best composite=0.5380  → {'n_estimators': 50, 'max_samples': 256, 'max_features': 0.8, 'contamination': 0.01, 'bootstr

STEP 5 : MODEL TRAINING

In [44]:
final_model = IsolationForest(random_state=42, n_jobs=-1, **best_params)
final_model.fit(X_scaled)

IsolationForest(contamination=0.01, max_features=0.8, max_samples=256,
                n_estimators=50, n_jobs=-1, random_state=42)

In [45]:
data["anomaly_label"] = final_model.predict(X_scaled)
data["anomaly_score"] = final_model.score_samples(X_scaled)
data["is_anomaly"]    = (data["anomaly_label"] == -1).astype(int)

In [46]:
total  = len(data)
n_anom = data["is_anomaly"].sum()
print(f"  Total observations : {total:,}")
print(f"  Anomalies detected : {n_anom:,}  ({100*n_anom/total:.2f}%)")
print(f"\n  Anomalies per source:")
print(data.groupby("source")["is_anomaly"]
      .agg(["sum","mean"])
      .rename(columns={"sum":"count","mean":"rate"})
      .assign(rate=lambda d: (d["rate"]*100).round(2))
      .sort_values("rate", ascending=False).to_string())

  Total observations : 67,740
  Anomalies detected : 678  (1.00%)

  Anomalies per source:
                                    count   rate
source                                          
ec2_disk_write_bytes_c0d644           433  10.74
ec2_disk_write_bytes_1ef3de           231   4.88
iio_us-east-1_i-a2eb1cd9_NetworkIn     12   0.97
ec2_network_in_257a54                   2   0.05
rds_cpu_utilization_cc0c53              0   0.00
grok_asg_anomaly                        0   0.00
elb_request_count_8c0756                0   0.00
ec2_network_in_5abac7                   0   0.00
ec2_cpu_utilization_24ae8d              0   0.00
ec2_cpu_utilization_53ea38              0   0.00
ec2_cpu_utilization_fe7f93              0   0.00
ec2_cpu_utilization_c6585a              0   0.00
ec2_cpu_utilization_ac20cd              0   0.00
ec2_cpu_utilization_825cc2              0   0.00
ec2_cpu_utilization_77c1ca              0   0.00
ec2_cpu_utilization_5f5533              0   0.00
rds_cpu_utilization_e47b3b 

STEP 6 : EVALUATION & VISUALISATION

In [51]:
# Plot 1 — Hyperparameter heatmaps
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Hyperparameter Tuning — Composite Score", fontsize=14, fontweight="bold")

for ax, (p1, p2) in zip(axes, [
        ("n_estimators","contamination"),
        ("max_samples","contamination"),
        ("max_features","contamination")]):
    
    pivot = results_df.groupby([p1, p2])["composite"].mean().unstack().fillna(0)
    
    im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
    
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(c) for c in pivot.columns], rotation=45)
    
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(i) for i in pivot.index])
    
    ax.set_xlabel(p2)
    ax.set_ylabel(p1)
    ax.set_title(f"{p1} vs {p2}")
    
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [54]:
# Plot 2 — Score distributions
normal  = data[data["is_anomaly"]==0]["anomaly_score"]
anomaly = data[data["is_anomaly"]==1]["anomaly_score"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Anomaly Score Distribution", fontsize=14, fontweight="bold")
ax1.hist(normal,  bins=50, alpha=0.7, color="#2196F3", label="Normal",  density=True)
ax1.hist(anomaly, bins=50, alpha=0.7, color="#F44336", label="Anomaly", density=True)
ax1.axvline(data["anomaly_score"].quantile(0.05), color="black", linestyle="--", linewidth=1.5, label="5th pct")
ax1.set_xlabel("Anomaly Score"); ax1.set_ylabel("Density")
ax1.set_title("Score Histogram"); ax1.legend()
ax2.boxplot([normal.values, anomaly.values],
            patch_artist=True,
            boxprops=dict(facecolor="#90CAF9"),
            medianprops=dict(color="#1565C0", linewidth=2))

ax2.set_xticks([1, 2])
ax2.set_xticklabels(["Normal", "Anomaly"])
ax2.set_ylabel("Anomaly Score")
ax2.set_title("Box-Plot by Label")

plt.tight_layout()

# Save + show (correct order)
plt.savefig(f"{OUTPUT_DIR}/02_score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [55]:
# Plot 3 — Per-source time-series with anomaly overlay
sources = sorted(data["source"].unique())
cols = 3; rows = (len(sources) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(22, rows*3.5))
axes = axes.flatten()
for i, src in enumerate(sources):
    ax  = axes[i]
    sub = data[data["source"]==src].sort_values("timestamp")
    anm = sub[sub["is_anomaly"]==1]
    ax.plot(sub["timestamp"], sub["value"], color="#1565C0", linewidth=0.8, alpha=0.8)
    ax.scatter(anm["timestamp"], anm["value"], color="#F44336", s=25, zorder=5,
               label=f"Anomaly ({len(anm)})")
    ax.set_title(src, fontsize=8, fontweight="bold")
    ax.tick_params(axis="x", labelsize=6, rotation=30)
    ax.tick_params(axis="y", labelsize=6)
    ax.legend(fontsize=6)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
fig.suptitle("Detected Anomalies — All Sources", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_anomalies_per_source.png", dpi=150, bbox_inches="tight")
plt.close(); print("  Saved: 03_anomalies_per_source.png")
 

  Saved: 03_anomalies_per_source.png


In [56]:
# Plot 4 — Feature importance (correlation with anomaly score)
feature_corr = pd.Series(
    [abs(np.corrcoef(X_scaled[:,j], data["anomaly_score"])[0,1]) for j in range(X_scaled.shape[1])],
    index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#F44336" if v > feature_corr.median() else "#2196F3" for v in feature_corr.values]
ax.barh(feature_corr.index, feature_corr.values, color=colors)
ax.set_xlabel("|Correlation with Anomaly Score|")
ax.set_title("Feature Importance", fontweight="bold")
ax.legend(handles=[Patch(facecolor="#F44336",label="Above median"),
                   Patch(facecolor="#2196F3",label="Below median")], loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close(); print("  Saved: 04_feature_importance.png")

  Saved: 04_feature_importance.png


In [57]:
# Plot 5 — CV stability
cv_rows = []
for fold_i, (tr, val) in enumerate(time_aware_folds(len(X_scaled), N_FOLDS)):
    model = IsolationForest(random_state=42, n_jobs=-1, **best_params)
    model.fit(X_scaled[tr])
    preds = model.predict(X_scaled[val])
    cv_rows.append({"fold": fold_i+1, "anomaly_rate": (preds==-1).mean()*100,
                    "train_size": len(tr), "val_size": len(val)})
cv_df = pd.DataFrame(cv_rows)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Cross-Validation Stability (Best Model)", fontweight="bold")
ax1.bar(cv_df["fold"], cv_df["anomaly_rate"], color="#7986CB")
ax1.axhline(cv_df["anomaly_rate"].mean(), color="#F44336", linestyle="--",
            linewidth=1.5, label=f'Mean {cv_df["anomaly_rate"].mean():.2f}%')
ax1.set_xlabel("Fold"); ax1.set_ylabel("Anomaly Rate (%)"); ax1.set_title("Rate per Fold"); ax1.legend()
ax2.bar(cv_df["fold"], cv_df["val_size"],   color="#80CBC4", label="Val size")
ax2.bar(cv_df["fold"], cv_df["train_size"], bottom=cv_df["val_size"],
        color="#B2DFDB", alpha=0.5, label="Train size")
ax2.set_xlabel("Fold"); ax2.set_ylabel("Samples"); ax2.set_title("Split per Fold"); ax2.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_cv_stability.png", dpi=150, bbox_inches="tight")
plt.close(); print("  Saved: 05_cv_stability.png")
 

  Saved: 05_cv_stability.png


In [58]:
data[["source","metric","timestamp","value","anomaly_score","is_anomaly"]]\
    .to_csv(f"{OUTPUT_DIR}/anomaly_results.csv", index=False)
print("  Saved: anomaly_results.csv")

  Saved: anomaly_results.csv
